In [9]:
import pandas as pd

viajes = "Trips.csv"
usuarios = "usuarios_ecobici_2026.csv"
estaciones = "nuevas-estaciones-bicicletas-publicas.csv"

df_viajes = pd.read_csv(viajes)
df_usuarios = pd.read_csv(usuarios)
df_estaciones = pd.read_csv(estaciones)



# Ver los nombres exactos de las columnas de cada tabla
print("Columnas de Viajes:", df_viajes.columns.tolist())
print("Columnas de Estaciones:", df_estaciones.columns.tolist())
print("Columnas de Usuarios:", df_usuarios.columns.tolist())



# Convertimos el texto a formato fecha de Pandas
df_viajes['fecha_origen_recorrido'] = pd.to_datetime(df_viajes['fecha_origen_recorrido'], errors='coerce')

# Creamos la tabla vacía para la Dimensión Tiempo
dim_tiempo = pd.DataFrame()

# Creamos el ID numérico único (formato AAAAMMDDHH)
dim_tiempo['id_tiempo'] = df_viajes['fecha_origen_recorrido'].dt.strftime('%Y%m%d%H').fillna(0).astype(int)

# Extraemos las jerarquías de tiempo
dim_tiempo['anio'] = df_viajes['fecha_origen_recorrido'].dt.year
dim_tiempo['mes'] = df_viajes['fecha_origen_recorrido'].dt.month
dim_tiempo['dia'] = df_viajes['fecha_origen_recorrido'].dt.day
dim_tiempo['hora'] = df_viajes['fecha_origen_recorrido'].dt.hour

# Borramos los duplicados para que quede un catálogo único de horas
dim_tiempo = dim_tiempo.drop_duplicates(subset=['id_tiempo']).reset_index(drop=True)

# Mostramos el resultado
print(dim_tiempo.head())



# Renombramos 'ID Cliente' para que coincida con la tabla de viajes
df_usuarios.rename(columns={'ID Cliente': 'id_usuario'}, inplace=True)

# Definimos los límites de los rangos y sus etiquetas para la edad
bins = [0, 18, 25, 35, 45, 60, 100]
etiquetas = ['Menores de 18', '18-25', '26-35', '36-45', '46-60', 'Mayores de 60']

# Creamos la nueva columna de Rangos Etarios usando 'edad_usuario'
df_usuarios['Rango_Etario'] = pd.cut(df_usuarios['edad_usuario'], bins=bins, labels=etiquetas, right=False)

# Mostramos el resultado
print(df_usuarios[['id_usuario', 'edad_usuario', 'Rango_Etario', 'genero_usuario']].head())




# Renombramos la columna 'id' a 'id_estacion'
df_estaciones.rename(columns={'id': 'id_estacion'}, inplace=True)

# Nos quedamos solo con las columnas que nos sirven como descriptores (Atributos)
dim_estacion = df_estaciones[['id_estacion', 'nombre', 'comuna', 'barrio', 'direccion', 'latitud', 'longitud']]

print(dim_estacion.head())


# 1. Recreamos el 'id_tiempo' en la tabla de viajes para que pueda "engancharse" con la Dimensión Tiempo
df_viajes['id_tiempo'] = df_viajes['fecha_origen_recorrido'].dt.strftime('%Y%m%d%H').fillna(0).astype(int)

# 2. Seleccionamos SOLO las Claves (Keys) y las Medidas (Measures) para la Tabla de Hechos
columnas_fact = [
    'Id_recorrido',
    'id_tiempo',
    'id_estacion_origen',
    'id_estacion_destino',
    'id_usuario',
    'duracion_recorrido'
]

fact_viajes = df_viajes[columnas_fact]

# Eliminamos registros con claves necesarias faltantes
fact_viajes = fact_viajes.dropna(
    subset=[
        'Id_recorrido',
        'id_tiempo',
        'id_estacion_origen',
        'id_estacion_destino',
        'id_usuario'
    ]
).reset_index(drop=True)

# Guardamos las tablas del Data Warehouse
fact_viajes.to_csv("DW_Fact_Viajes.csv", index=False)
dim_tiempo.to_csv("DW_Dim_Tiempo.csv", index=False)
df_usuarios.to_csv("DW_Dim_Usuarios.csv", index=False)
dim_estacion.to_csv("DW_Dim_Estaciones.csv", index=False)



FileNotFoundError: [Errno 2] No such file or directory: 'Trips.csv'